In [1]:
!git clone https://github.com/BorgwardtLab/SAT.git
%cd SAT
!ls

Cloning into 'SAT'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 91 (delta 38), reused 71 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 2.63 MiB | 17.96 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/kaggle/working/SAT
experiments  images  LICENSE  pretrained_models  README.md  s  sat


In [2]:
!ls sat/
!ls experiments/

data.py        __init__.py  metric.py  position_encoding.py
gnn_layers.py  layers.py    models.py  utils.py
model_visu.py	train_ppa.py   train_zinc.py
train_code2.py	train_SBMs.py  utils.py


In [3]:
import torch
print(torch.__version__)
print(torch.version.cuda)

2.10.0+cu128
12.8


In [4]:
!pip install torch_geometric -q
!pip install ogb -q
!pip install performer-pytorch -q
!pip install einops -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 2.9 MB/s eta 0:00:00


In [5]:
# Skip pyg_lib — not available for torch 2.10 yet
# Install scatter/sparse from the closest available wheel
!pip install torch_scatter torch_sparse \
    -f https://data.pyg.org/whl/torch-2.4.0+cu124.html -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 81.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 106.4 MB/s eta 0:00:00


In [6]:
!pip install torch_geometric -q
!pip install ogb -q  
!pip install performer-pytorch -q
!pip install einops -q
# No pyg_lib or torch_scatter needed — PyG uses torch fallbacks

In [7]:
import torch
import torch_geometric
print("PyTorch:", torch.__version__)
print("PyG:", torch_geometric.__version__)
print("CUDA:", torch.cuda.is_available())

# Test scatter works
from torch_geometric.utils import scatter
print("Scatter: OK ✅")

/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_sparse/_version_cuda.so
  import torch_geometric.typing


PyTorch: 2.10.0+cu128
PyG: 2.7.0
CUDA: True
Scatter: OK ✅


In [8]:
!head -80 /kaggle/working/SAT/experiments/train_zinc.py

# -*- coding: utf-8 -*-
import os
import copy
import argparse
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
from torch import nn, optim
import torch.nn.functional as F
from torch_geometric.data import DataLoader
from torch_geometric import datasets
import torch_geometric.utils as utils
from sat.models import GraphTransformer
from sat.data import GraphDataset
from sat.utils import count_parameters
from sat.position_encoding import POSENCODINGS
from sat.gnn_layers import GNN_TYPES
from timeit import default_timer as timer


def load_args():
    parser = argparse.ArgumentParser(
        description='Structure-Aware Transformer on ZINC',
        formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    parser.add_argument('--seed', type=int, default=0,
                        help='random seed')
    parser.add_argument('--dataset', type=str, default="ZINC",
                        help='name of dataset')
    parser.add_argument('--num-heads', ty

In [9]:
!grep -n "GNN_TYPES" /kaggle/working/SAT/sat/gnn_layers.py | head -5

17:GNN_TYPES = [
24:EDGE_GNN_TYPES = [


In [10]:
!grep "GNN_TYPES" /kaggle/working/SAT/sat/gnn_layers.py

GNN_TYPES = [
EDGE_GNN_TYPES = [


In [11]:
!sed -n '17,35p' /kaggle/working/SAT/sat/gnn_layers.py

GNN_TYPES = [
    'graph', 'graphsage', 'gcn',
    'gin', 'gine',
    'pna', 'pna2', 'pna3', 'mpnn', 'pna4',
    'rwgnn', 'khopgnn'
]

EDGE_GNN_TYPES = [
    'gine', 'gcn',
    'pna', 'pna2', 'pna3', 'mpnn', 'pna4'
]


def get_simple_gnn_layer(gnn_type, embed_dim, **kwargs):
    edge_dim = kwargs.get('edge_dim', None)
    if gnn_type == "graph":
        return gnn.GraphConv(embed_dim, embed_dim)
    elif gnn_type == "graphsage":
        return gnn.SAGEConv(embed_dim, embed_dim)


In [12]:
!sed -n '30,80p' /kaggle/working/SAT/sat/gnn_layers.py

def get_simple_gnn_layer(gnn_type, embed_dim, **kwargs):
    edge_dim = kwargs.get('edge_dim', None)
    if gnn_type == "graph":
        return gnn.GraphConv(embed_dim, embed_dim)
    elif gnn_type == "graphsage":
        return gnn.SAGEConv(embed_dim, embed_dim)
    elif gnn_type == "gcn":
        if edge_dim is None:
            return gnn.GCNConv(embed_dim, embed_dim)
        else:
            return GCNConv(embed_dim, edge_dim)
    elif gnn_type == "gin":
        mlp = mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(True),
            nn.Linear(embed_dim, embed_dim),
        )
        return gnn.GINConv(mlp, train_eps=True)
    elif gnn_type == "gine":
        mlp = mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(True),
            nn.Linear(embed_dim, embed_dim),
        )
        return gnn.GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
    elif gnn_type == "pna":
        aggregators = ['mean', 'min', '

In [13]:
!cd /kaggle/working/SAT && python experiments/train_zinc.py \
    --seed 0 \
    --gnn-type gin \
    --k-hop 1 \
    --se gnn \
    --abs-pe rw \
    --abs-pe-dim 20 \
    --epochs 5 \
    --batch-size 128 \
    --dropout 0.2 \
    --num-layers 6 \
    --num-heads 8 \
    --dim-hidden 64

/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_sparse/_version_cuda.so
  import torch_geometric.typing
Traceback (most recent call last):
  File "/kaggle/working/SAT/experiments/train_zinc.py", line 15, in <module>
    from sat.models import GraphTransformer
ModuleNotFoundError: No module named 'sat'


In [14]:
import sys
sys.path.insert(0, '/kaggle/working/SAT')

# Reinstall scatter/sparse for torch 2.10 + cu128
!pip install torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.4.0+cu124.html -q

In [15]:
!cd /kaggle/working/SAT && PYTHONPATH=/kaggle/working/SAT python experiments/train_zinc.py \
    --seed 0 \
    --gnn-type gin \
    --k-hop 1 \
    --se gnn \
    --abs-pe rw \
    --abs-pe-dim 20 \
    --epochs 5 \
    --batch-size 128 \
    --dropout 0.2 \
    --num-layers 6 \
    --num-heads 8 \
    --dim-hidden 64

/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_sparse/_version_cuda.so
  import torch_geometric.typing
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/_ops.py", line 1442, in load_library
    ctypes.CDLL(path)
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cuda.so: undefined sy

In [16]:
import torch

# 1. Format the exact version strings needed by PyG
# PyG hosts wheels based on minor versions (e.g., 2.1.0, 2.4.0)
pt_version = '.'.join(torch.__version__.split('+')[0].split('.')[:2]) + '.0'
cu_version = torch.version.cuda.replace('.', '')
wheel_url = f"https://data.pyg.org/whl/torch-{pt_version}+cu{cu_version}.html"

print(f"Targeting exact PyG wheels for your environment: {wheel_url}")

# 2. Wipe the broken installation
!pip uninstall -y torch-scatter torch-sparse

# 3. Install the perfect match
!pip install torch-scatter torch-sparse -f {wheel_url} -q
print("Clean installation complete! ✅")

Targeting exact PyG wheels for your environment: https://data.pyg.org/whl/torch-2.10.0+cu128.html
Found existing installation: torch_scatter 2.1.2+pt24cu124
Uninstalling torch_scatter-2.1.2+pt24cu124:
  Successfully uninstalled torch_scatter-2.1.2+pt24cu124
Found existing installation: torch_sparse 0.6.18+pt24cu124
Uninstalling torch_sparse-0.6.18+pt24cu124:
  Successfully uninstalled torch_sparse-0.6.18+pt24cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 84.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 113.0 MB/s eta 0:00:00
Clean installation complete! ✅


In [17]:
# Clear any lingering import issues by reloading the path
import sys
if '/kaggle/working/SAT' not in sys.path:
    sys.path.insert(0, '/kaggle/working/SAT')

# Run the 5-epoch test
!cd /kaggle/working/SAT && PYTHONPATH=/kaggle/working/SAT python experiments/train_zinc.py \
    --seed 0 \
    --gnn-type gin \
    --k-hop 1 \
    --se gnn \
    --abs-pe rw \
    --abs-pe-dim 20 \
    --epochs 5 \
    --batch-size 128 \
    --dropout 0.2 \
    --num-layers 6 \
    --num-heads 8 \
    --dim-hidden 64

Namespace(seed=0, dataset='ZINC', num_heads=8, num_layers=6, dim_hidden=64, dropout=0.2, epochs=5, lr=0.001, weight_decay=1e-05, batch_size=128, abs_pe='rw', abs_pe_dim=20, outdir='', warmup=5000, layer_norm=False, use_edge_attr=False, edge_dim=32, gnn_type='gin', k_hop=1, global_pool='mean', se='gnn', use_cuda=True, batch_norm=True, save_logs=False)
Extracting ../datasets/ZINC/molecules.zip
Processing...
Processing test dataset: 100%|████████████| 1000/1000 [00:00<00:00, 8328.58it/s]
Done!
/kaggle/working/SAT/experiments/train_zinc.py:207: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_dset, batch_size=args.batch_size,
Data(x=[29], edge_index=[2, 64], edge_attr=[64], y=[1, 1], complete_edge_index=[2, 841], degree=[29])
/kaggle/working/SAT/experiments/train_zinc.py:216: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader = DataLoader(val_dset, batch_size=args.batch_size, shuffle=Fals

In [18]:
# Patch for PyTorch 2.x Compatibility
file_path = '/kaggle/working/SAT/sat/models.py'

with open(file_path, 'r') as file:
    code = file.read()

# The missing property PyTorch 2.10 expects
target_line = "self.encoder = GraphTransformerEncoder(encoder_layer, num_layers)"
patched_line = """encoder_layer.self_attn.batch_first = False
        self.encoder = GraphTransformerEncoder(encoder_layer, num_layers)"""

# Apply and save the patch
if "batch_first" not in code:
    code = code.replace(target_line, patched_line)
    with open(file_path, 'w') as file:
        file.write(code)
    print("PyTorch 2.x Patch applied successfully! ✅")
else:
    print("Patch was already applied.")

PyTorch 2.x Patch applied successfully! ✅


In [19]:
!cd /kaggle/working/SAT && PYTHONPATH=/kaggle/working/SAT python experiments/train_zinc.py \
    --seed 0 \
    --gnn-type gin \
    --k-hop 1 \
    --se gnn \
    --abs-pe rw \
    --abs-pe-dim 20 \
    --epochs 5 \
    --batch-size 128 \
    --dropout 0.2 \
    --num-layers 6 \
    --num-heads 8 \
    --dim-hidden 64

Namespace(seed=0, dataset='ZINC', num_heads=8, num_layers=6, dim_hidden=64, dropout=0.2, epochs=5, lr=0.001, weight_decay=1e-05, batch_size=128, abs_pe='rw', abs_pe_dim=20, outdir='', warmup=5000, layer_norm=False, use_edge_attr=False, edge_dim=32, gnn_type='gin', k_hop=1, global_pool='mean', se='gnn', use_cuda=True, batch_norm=True, save_logs=False)
/kaggle/working/SAT/experiments/train_zinc.py:207: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_dset, batch_size=args.batch_size,
Data(x=[29], edge_index=[2, 64], edge_attr=[64], y=[1, 1], complete_edge_index=[2, 841], degree=[29])
/kaggle/working/SAT/experiments/train_zinc.py:216: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader = DataLoader(val_dset, batch_size=args.batch_size, shuffle=False)
/kaggle/working/SAT/sat/models.py:76: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.s

In [20]:
!cd /kaggle/working/SAT && PYTHONPATH=/kaggle/working/SAT python experiments/train_zinc.py \
    --seed 0 \
    --gnn-type gin \
    --k-hop 3 \
    --se gnn \
    --abs-pe rw \
    --abs-pe-dim 20 \
    --epochs 2000 \
    --batch-size 128 \
    --lr 0.001 \
    --weight-decay 1e-5 \
    --dropout 0.2 \
    --num-layers 6 \
    --num-heads 8 \
    --dim-hidden 64 \
    --outdir /kaggle/working/results

Namespace(seed=0, dataset='ZINC', num_heads=8, num_layers=6, dim_hidden=64, dropout=0.2, epochs=2000, lr=0.001, weight_decay=1e-05, batch_size=128, abs_pe='rw', abs_pe_dim=20, outdir='/kaggle/working/results/ZINC/seed0/rw_20/gin_3_0.2_0.001_1e-05_6_8_64_BN', warmup=5000, layer_norm=False, use_edge_attr=False, edge_dim=32, gnn_type='gin', k_hop=3, global_pool='mean', se='gnn', use_cuda=True, batch_norm=True, save_logs=True)
/kaggle/working/SAT/experiments/train_zinc.py:207: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_dset, batch_size=args.batch_size,
Data(x=[29], edge_index=[2, 64], edge_attr=[64], y=[1, 1], complete_edge_index=[2, 841], degree=[29])
/kaggle/working/SAT/experiments/train_zinc.py:216: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader = DataLoader(val_dset, batch_size=args.batch_size, shuffle=False)
/kaggle/working/SAT/sat/models.py:76: UserWarning: enable_nested_t